# 06 — Pipeline de Ibagué (02 + 03) en Villavicencio y Montería

Ejecuta **copias** de `02_clima_canal_endemico_ibague_3` y `03_modelos_ibague_1` cambiando solo el municipio. Los originales no se tocan.

| Municipio | COD_MUN | Clima (GEE, diario 2010–2025) |
|---|---|---|
| Villavicencio | 50001 | `Datos/clima/GEE_Clima_Villavicencio` |
| Montería | 23001 | `Datos/clima/GEE_Clima_Monteria` |

Cambios que se aplican a cada copia (nada más):
- `COD_MUN`, `MUNICIPIO`, `CLIMA_DIR` y `POBLACION` del municipio.
- En el 03, la carpeta de modelos pasa de `procesado/modelos/` a `procesado/modelos_{COD_MUN}/`. **Sin esto el 03 sobrescribe los modelos y predicciones de Ibagué.**
- `drive.mount` no se repite (Drive ya está montado aquí).

Salidas:
- Notebooks ejecutados, con salidas: `Colab Notebooks/02_clima_canal_endemico_{villavicencio|monteria}.ipynb` y `03_modelos_{villavicencio|monteria}.ipynb`.
- Datos: `procesado/serie_nivel2_{cod}.csv`, `tabla_modelado_{cod}.csv`, `comparacion_modelos_{cod}.csv`, `comparacion_clima_{cod}.csv`, `modelos_{cod}/`.

Secciones 5–8: comparación con las predicciones del notebook 05 (modelos globales, sin clima local, ventana creciente 2016–2025), cobertura del canal en los 39 municipios y métricas globales del 05 recalculadas desde `predicciones_test.csv` (el 05 del 20-nov guardó las predicciones pero no las métricas). Las tablas recalculadas quedan como `metricas_globales_desde_predicciones.csv` y `pruebas_pareadas_desde_predicciones.csv` en `comparacion_dl_profundo/mensual/`.

Ojo al leer: el test 2023–2025 de cada municipio tiene muy pocos inicios de brote. Sirve para ver si el patrón de Ibagué se repite, no para elegir modelo.

In [1]:
# 1. Configuración
import os, re, sys, time, json, subprocess
from google.colab import drive
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'nbclient', 'nbformat', 'ipykernel'], check=True)
import nbformat
from nbclient import NotebookClient
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, average_precision_score

NB = '/content/drive/MyDrive/Colab Notebooks'
BASE = '/content/drive/MyDrive/Datos'
RUTA = f'{BASE}/dengue/procesado'
ORIG02 = f'{NB}/02_clima_canal_endemico_ibague_3.ipynb'
ORIG03 = f'{NB}/03_modelos_ibague_1.ipynb'
PRED05 = f'{BASE}/dengue/comparacion_dl_profundo/mensual/predicciones_test.csv'
MUNIS = {'villavicencio': ('50001', 'Villavicencio', 'GEE_Clima_Villavicencio'),
         'monteria': ('23001', 'Montería', 'GEE_Clima_Monteria')}

for p in (ORIG02, ORIG03):
    assert os.path.exists(p), f'No encuentro {p}'
for slug, (cod, nombre, carpeta) in MUNIS.items():
    d = f'{BASE}/clima/{carpeta}'
    n = len([f for f in os.listdir(d) if f.endswith('.csv')]) if os.path.isdir(d) else 0
    print(f'{nombre}: {n} CSV de clima en {d}')
    assert n > 0, f'Falta el acceso directo {d}'

Mounted at /content/drive
Villavicencio: 16 CSV de clima en /content/drive/MyDrive/Datos/clima/GEE_Clima_Villavicencio
Montería: 16 CSV de clima en /content/drive/MyDrive/Datos/clima/GEE_Clima_Monteria


In [2]:
# 2. Copia parametrizada de cada notebook
PARCHE = ("try:\n"
          "    import google.colab.drive as _d\n"
          "    _d.mount = lambda *a, **k: print('Drive ya está montado (notebook 06)')\n"
          "except Exception as e:\n"
          "    print('sin google.colab:', e)")

def parametrizar(nb, cod, nombre, carpeta, slug):
    cambios = [
        ('COD_MUN', r"^(\s*COD_MUN\s*=\s*)'\d{5}'", rf"\g<1>'{cod}'"),
        ('MUNICIPIO', r"^(\s*MUNICIPIO\s*=\s*)'[^']*'", rf"\g<1>'{nombre}'"),
        ('CLIMA_DIR', r"^(\s*CLIMA_DIR\s*=\s*)f?'[^']*'", rf"\g<1>f'{{BASE}}/clima/{carpeta}'"),
        ('POBLACION', r"^(\s*POBLACION\s*=\s*)f?'[^']*'", rf"\g<1>f'{{BASE}}/poblacion_{slug}.csv'"),
        ('SAL', r"^(\s*SAL\s*=\s*)f'\{RUTA\}/modelos'", r"\g<1>f'{RUTA}/modelos_{COD_MUN}'"),
    ]
    hechos = {nombre_c: 0 for nombre_c, _, _ in cambios}
    for c in nb.cells:
        if c.cell_type != 'code':
            continue
        for nombre_c, pat, rep in cambios:
            c.source, n = re.subn(pat, rep, c.source, flags=re.M)
            hechos[nombre_c] += n
        c.source = c.source.replace("'Ibagué — canal endémico'", f"'{nombre} — canal endémico'")
    nb.cells.insert(0, nbformat.v4.new_code_cell(PARCHE))
    restos = [l.strip() for c in nb.cells if c.cell_type == 'code' for l in c.source.splitlines()
              if '73001' in l or 'GEE_Clima_Ibague' in l or 'poblacion_ibague' in l]
    return nb, hechos, restos

def ejecutar(orig, destino, cod, nombre, carpeta, slug, es_03=False):
    nb = nbformat.read(orig, as_version=4)
    nb, hechos, restos = parametrizar(nb, cod, nombre, carpeta, slug)
    print(f'  {os.path.basename(orig)} → cambios aplicados: {hechos}')
    if restos:
        raise RuntimeError(f'Quedan referencias a Ibagué: {restos[:5]}')
    if hechos['COD_MUN'] != 1:
        raise RuntimeError('No encontré la línea COD_MUN')
    if es_03 and hechos['SAL'] != 1:
        raise RuntimeError('No pude redirigir procesado/modelos/: se sobrescribirían los archivos de Ibagué')
    t0, estado = time.time(), 'ok'
    try:
        NotebookClient(nb, timeout=1800, kernel_name='python3',
                       resources={'metadata': {'path': '/content'}}).execute()
    except Exception as e:
        estado = f'ERROR {type(e).__name__}: {str(e)[-800:]}'
    nbformat.write(nb, destino)
    print(f'  {os.path.basename(destino)}: {estado} ({time.time() - t0:.0f} s)')
    return estado

In [3]:
# 3. Correr 02 y 03 para cada municipio
ESTADO = {}
for slug, (cod, nombre, carpeta) in MUNIS.items():
    print(f'\n===== {nombre} ({cod}) =====')
    e2, e3 = '—', '—'
    try:
        e2 = ejecutar(ORIG02, f'{NB}/02_clima_canal_endemico_{slug}.ipynb', cod, nombre, carpeta, slug)
        if e2 == 'ok':
            t = pd.read_csv(f'{RUTA}/tabla_modelado_{cod}.csv')
            n_split = t['split'].value_counts().reindex(['train', 'val', 'test'], fill_value=0).to_dict()
            print('  filas por partición:', n_split)
            if min(n_split.values()) == 0:
                e3 = f'no se corre: partición vacía {n_split} (el canal no tiene umbral para esos años)'
                print('  03:', e3)
            else:
                e3 = ejecutar(ORIG03, f'{NB}/03_modelos_{slug}.ipynb', cod, nombre, carpeta, slug, es_03=True)
    except Exception as e:
        print('  ', type(e).__name__, e)
        e2 = e2 if e2 != '—' else f'ERROR {e}'
    ESTADO[nombre] = {'02': e2, '03': e3}
print(json.dumps(ESTADO, indent=1, ensure_ascii=False))


===== Villavicencio (50001) =====
  02_clima_canal_endemico_ibague_3.ipynb → cambios aplicados: {'COD_MUN': 1, 'MUNICIPIO': 1, 'CLIMA_DIR': 1, 'POBLACION': 1, 'SAL': 0}
  02_clima_canal_endemico_villavicencio.ipynb: ok (20 s)
  filas por partición: {'train': 82, 'val': 48, 'test': 36}
  03_modelos_ibague_1.ipynb → cambios aplicados: {'COD_MUN': 1, 'MUNICIPIO': 0, 'CLIMA_DIR': 0, 'POBLACION': 0, 'SAL': 1}
  03_modelos_villavicencio.ipynb: ok (92 s)

===== Montería (23001) =====
  02_clima_canal_endemico_ibague_3.ipynb → cambios aplicados: {'COD_MUN': 1, 'MUNICIPIO': 1, 'CLIMA_DIR': 1, 'POBLACION': 1, 'SAL': 0}
  02_clima_canal_endemico_monteria.ipynb: ok (10 s)
  filas por partición: {'train': 84, 'val': 36, 'test': 0}
  03: no se corre: partición vacía {'train': 84, 'val': 36, 'test': 0} (el canal no tiene umbral para esos años)
{
 "Villavicencio": {
  "02": "ok",
  "03": "ok"
 },
 "Montería": {
  "02": "ok",
  "03": "no se corre: partición vacía {'train': 84, 'val': 36, 'test': 0} (e

In [4]:
# 4. Canal endémico y resultados de cada municipio (Ibagué como referencia)
def resumen(cod, nombre):
    p2 = f'{RUTA}/serie_nivel2_{cod}.csv'
    if not os.path.exists(p2):
        print(f'\n##### {nombre}: no existe {p2}'); return
    n2 = pd.read_csv(p2, parse_dates=['mes']).set_index('mes')
    ex = n2['exceso'].dropna().loc['2012':]
    print(f'\n##### {nombre} ({cod})')
    print('Casos por año:', n2['casos'].groupby(n2.index.year).sum().loc[2012:].astype(int).to_dict())
    print('Proporción de meses en exceso por año:', ex.groupby(ex.index.year).mean().round(2).to_dict())
    print(f'Meses en exceso 2012–2025: {ex.mean():.0%} | meses con lluvia (2010+): {n2.loc["2010":, "lluvia_mm"].notna().mean():.0%}'
          f' | temperatura media: {n2["temp_c"].mean():.1f} °C')
    for f in (f'comparacion_modelos_{cod}.csv', f'comparacion_clima_{cod}.csv'):
        p = f'{RUTA}/{f}'
        print(f'\n{f}:')
        print(pd.read_csv(p).to_string(index=False) if os.path.exists(p) else '  no existe (el 03 no terminó)')

for cod, nombre in [('73001', 'Ibagué')] + [(c, n) for c, n, _ in MUNIS.values()]:
    resumen(cod, nombre)


##### Ibagué (73001)
Casos por año: {2012: 2326, 2013: 5335, 2014: 3047, 2015: 5082, 2016: 2200, 2017: 603, 2018: 600, 2019: 5197, 2020: 2698, 2021: 317, 2022: 343, 2023: 4147, 2024: 13327, 2025: 1386}
Proporción de meses en exceso por año: {2012: 0.42, 2013: 0.92, 2014: 0.75, 2015: 1.0, 2016: 0.33, 2017: 0.0, 2018: 0.0, 2019: 0.67, 2020: 0.25, 2021: 0.0, 2022: 0.0, 2023: 0.5, 2024: 1.0, 2025: 0.0}
Meses en exceso 2012–2025: 42% | meses con lluvia (2010+): 100% | temperatura media: 17.7 °C

comparacion_modelos_73001.csv:
           modelo conjunto  recall  precision   f1  pr_auc  recall_inicios  n_inicios
     Persistencia      val    0.82       0.82 0.82     NaN             0.0        2.0
     Persistencia     test    0.89       0.89 0.89     NaN             0.0        2.0
        Logística      val    0.82       0.90 0.86    0.93             0.0        2.0
        Logística     test    1.00       0.67 0.80    0.92             1.0        2.0
          XGBoost      val    1.00       0

In [5]:
# 5. Comparación con el 05 (modelos globales, 39 municipios, sin clima local, ventana creciente)
if not os.path.exists(PRED05):
    print('No está', PRED05, '→ correr primero el 05')
else:
    print('Predicciones del 05 modificadas:', time.ctime(os.path.getmtime(PRED05)))
    pr = pd.read_csv(PRED05, dtype={'mun': str}, parse_dates=['fecha_t'])
    modelos = [c[2:] for c in pr.columns if c.startswith('p|')]

    def met(g, m):
        y, p = g['y'].values, g[f'p|{m}'].values
        pred = (p >= g[f'u|{m}'].values).astype(int)
        ini = (y == 1) & (g['exc_o'].values == 0)
        return {'f1': f1_score(y, pred, zero_division=0),
                'pr_auc': average_precision_score(y, p) if 0 < y.sum() < len(y) else np.nan,
                'recall_inicios': pred[ini].mean() if ini.any() else np.nan,
                'n': len(y), 'excesos': int(y.sum()), 'inicios': int(ini.sum())}

    for cod, nombre in [('73001', 'Ibagué')] + [(c, n) for c, n, _ in MUNIS.values()]:
        g = pr[pr['mun'] == cod]
        if g.empty:
            print(f'\n{nombre} ({cod}) no está entre los municipios del 05'); continue
        for etiqueta, sub in [('2016–2025', g), ('2023–2025 (mismo periodo que el test del 03)', g[g['fecha_t'].dt.year >= 2023])]:
            print(f'\n{nombre} — 05 global, {etiqueta}')
            print(pd.DataFrame({m: met(sub, m) for m in modelos}).T.round(3).to_string())
        p2 = f'{RUTA}/serie_nivel2_{cod}.csv'
        if os.path.exists(p2):
            e02 = pd.read_csv(p2, parse_dates=['mes']).set_index('mes')['exceso'].rename('exceso_02')
            comun = g.set_index('fecha_t')[['y']].join(e02, how='inner').dropna()
            print(f'Etiqueta de exceso igual en 05 y 02: {(comun["y"] == comun["exceso_02"]).mean():.1%} de {len(comun)} meses')

Predicciones del 05 modificadas: Fri Nov 20 23:50:50 2026

Ibagué — 05 global, 2016–2025
                            f1  pr_auc  recall_inicios      n  excesos  inicios
Persistencia             0.812   0.703             0.0  118.0     31.0      5.0
Binomial negativa local  0.817   0.889             0.8  118.0     31.0      5.0
LightGBM global          0.829   0.932             0.6  118.0     31.0      5.0
GRU global (1 capa)      0.853   0.949             0.6  118.0     31.0      5.0
LSTM global (2 capas)    0.841   0.916             0.6  118.0     31.0      5.0
TCN global               0.824   0.916             0.4  118.0     31.0      5.0

Ibagué — 05 global, 2023–2025 (mismo periodo que el test del 03)
                            f1  pr_auc  recall_inicios     n  excesos  inicios
Persistencia             0.889   0.846             0.0  36.0     18.0      2.0
Binomial negativa local  0.944   0.994             1.0  36.0     18.0      2.0
LightGBM global          0.923   0.977          

In [6]:
# 6. ¿En cuántos de los 39 municipios del 05 el canal deja años sin etiqueta? (caso Montería)
pr = pd.read_csv(PRED05, dtype={'mun': str}, parse_dates=['fecha_t'])
cob = (pr.assign(anio_t=pr['fecha_t'].dt.year).groupby('mun')['anio_t']
         .agg(meses='size', primer_anio='min', ultimo_anio='max'))
try:
    nombres = (pd.read_csv(f'{RUTA}/serie_mensual_municipios.csv', dtype={'cod_mun': str},
                           usecols=['cod_mun', 'DEPARTAMENTO_OCURRENCIA', 'MUNICIPIO_OCURRENCIA'])
                 .drop_duplicates('cod_mun').set_index('cod_mun'))
    cob = cob.join(nombres)
except Exception as e:
    print('sin nombres de municipio:', e)
maximo = cob['meses'].max()
cortos = cob[cob['meses'] < maximo - 2].sort_values('meses')
print(f'{len(cob)} municipios | meses de prueba por municipio: máximo {maximo}, mediana {cob.meses.median():.0f}')
print(f'Con años sin canal (menos de {maximo - 2} meses de prueba): {len(cortos)}')
print(cortos.to_string())

39 municipios | meses de prueba por municipio: máximo 118, mediana 118
Con años sin canal (menos de 116 meses de prueba): 5
       meses  primer_anio  ultimo_anio DEPARTAMENTO_OCURRENCIA MUNICIPIO_OCURRENCIA
mun                                                                                
23001     70         2016         2021                 CORDOBA             MONTERIA
54498    106         2016         2024         NORTE SANTANDER                OCAÑA
76001    106         2016         2024                   VALLE                 CALI
76834    106         2016         2024                   VALLE                TULUA
20011    107         2017         2025                   CESAR            AGUACHICA


In [7]:
# 7. Resultados globales que guardó el 05 (comparacion_dl_profundo/mensual)
D05 = os.path.dirname(PRED05)
for f in ['metricas_globales.csv', 'pruebas_pareadas.csv', 'sensibilidad.csv', 'resumen.json']:
    p = f'{D05}/{f}'
    if not os.path.exists(p):
        print(f'\n{f}: no existe'); continue
    print(f'\n===== {f} (modificado {time.ctime(os.path.getmtime(p))})')
    if f.endswith('.json'):
        r = json.load(open(p))
        print({k: r.get(k) for k in ('frecuencia', 'rapido', 'municipios', 'filas_prueba', 'inicios_en_prueba',
                                     'oni', 'parametros_red_elegida_mediana', 'segundos_por_modelo')})
        continue
    d = pd.read_csv(p)
    if f == 'pruebas_pareadas.csv':
        d = d[[c for c in ('A', 'B', 'dif_F1 (A-B)', 'IC95_F1', 'dif_PR-AUC (A-B)', 'IC95_PR-AUC',
                           'Brier mejor en A (% bloques)', 'p Holm Brier', 'p Holm PR-AUC') if c in d.columns]]
    with pd.option_context('display.width', 250, 'display.max_columns', 30):
        print(d.round(3).to_string(index=False))


metricas_globales.csv: no existe

pruebas_pareadas.csv: no existe

sensibilidad.csv: no existe

resumen.json: no existe


In [8]:
# 8. Métricas globales y pruebas pareadas del 05, recalculadas desde predicciones_test.csv
# El 05 guardó las predicciones (sección 11) pero no las métricas (secciones 12, 13 y 16).
# Se usa el mismo código del 05 (misma semilla del bootstrap). No se reentrena nada.
from scipy import stats
from sklearn.metrics import precision_score, recall_score, brier_score_loss

PR = pd.read_csv(PRED05, dtype={'mun': str}, parse_dates=['fecha_t'])
NOMBRES = [c[2:] for c in PR.columns if c.startswith('p|')]
DL = [n for n in NOMBRES if any(k in n for k in ('GRU', 'LSTM', 'TCN'))]

def evaluar(g, pred, prob):
    y = g['y'].values; pred = np.asarray(pred).astype(int)
    ini = (y == 1) & (g['exc_o'].values == 0)
    neg = y == 0
    return {'f1': f1_score(y, pred, zero_division=0), 'pr_auc': average_precision_score(y, prob),
            'recall': recall_score(y, pred, zero_division=0), 'precision': precision_score(y, pred, zero_division=0),
            'recall_inicios': pred[ini].mean() if ini.any() else np.nan, 'n_inicios': int(ini.sum()),
            'falsas_alarmas': pred[neg].mean() if neg.any() else np.nan,
            'brier': brier_score_loss(y, prob), 'n': len(y), 'positivos': int(y.sum())}

def evaluar_modelo(g, n):
    p, u = g[f'p|{n}'].values, g[f'u|{n}'].values
    return evaluar(g, p >= u, p)

res = pd.DataFrame({n: evaluar_modelo(PR, n) for n in NOMBRES}).T
print(f'{PR.mun.nunique()} municipios | {len(PR):,} periodos de prueba | {int(res.loc["Persistencia", "n_inicios"])} inicios de brote')
print('falsas_alarmas = proporción de meses SIN exceso en los que el modelo alerta')
print(res[['f1', 'pr_auc', 'recall', 'precision', 'recall_inicios', 'falsas_alarmas', 'brier']].round(3).to_string())

def por_bloque(nombre):
    out = []
    for (m, Y), g in PR.groupby(['mun', 'anio_o']):
        p = g[f'p|{nombre}'].values; y = g['y'].values
        out.append({'mun': m, 'anio': Y, 'brier': np.mean((p - y) ** 2),
                    'pr_auc': average_precision_score(y, p) if 0 < y.sum() < len(y) else np.nan})
    return pd.DataFrame(out).set_index(['mun', 'anio'])

BLOQ = {n: por_bloque(n) for n in NOMBRES}
bloques = list(PR.groupby(['mun', 'anio_o']).indices.values())
rng = np.random.default_rng(42)
muestras = [np.concatenate([bloques[i] for i in rng.integers(0, len(bloques), len(bloques))]) for _ in range(2000)]

def comparar(a, b):
    da, db = BLOQ[a], BLOQ[b]
    w_br = stats.wilcoxon(da['brier'], db['brier'])
    ok = da['pr_auc'].notna() & db['pr_auc'].notna()
    w_ap = stats.wilcoxon(da.loc[ok, 'pr_auc'], db.loc[ok, 'pr_auc'])
    ya = PR['y'].values
    pa, pb = PR[f'p|{a}'].values, PR[f'p|{b}'].values
    ea, eb = (pa >= PR[f'u|{a}'].values).astype(int), (pb >= PR[f'u|{b}'].values).astype(int)
    d_f1, d_ap = [], []
    for idx in muestras:
        d_f1.append(f1_score(ya[idx], ea[idx]) - f1_score(ya[idx], eb[idx]))
        d_ap.append(average_precision_score(ya[idx], pa[idx]) - average_precision_score(ya[idx], pb[idx]))
    return {'A': a, 'B': b,
            'dif_F1 (A-B)': f1_score(ya, ea) - f1_score(ya, eb), 'IC95_F1': np.round(np.percentile(d_f1, [2.5, 97.5]), 3),
            'dif_PR-AUC (A-B)': average_precision_score(ya, pa) - average_precision_score(ya, pb),
            'IC95_PR-AUC': np.round(np.percentile(d_ap, [2.5, 97.5]), 3),
            'Brier mejor en A (% bloques)': (da['brier'] < db['brier']).mean(), 'p Wilcoxon Brier': w_br.pvalue,
            'PR-AUC mejor en A (% bloques)': (da.loc[ok, 'pr_auc'] > db.loc[ok, 'pr_auc']).mean(),
            'p Wilcoxon PR-AUC': w_ap.pvalue, 'bloques': len(da), 'bloques PR-AUC': int(ok.sum())}

def holm(p):
    p = np.asarray(p, float); m = len(p); o = np.argsort(p)
    adj = np.minimum(np.maximum.accumulate((m - np.arange(m)) * p[o]), 1.0)
    out = np.empty(m); out[o] = adj
    return out

PROFUNDAS = [n for n in DL if n != 'GRU global (1 capa)']
pares = ([('LightGBM global', n) for n in DL]
         + [(n, 'GRU global (1 capa)') for n in PROFUNDAS]
         + [(n, 'Binomial negativa local') for n in ['LightGBM global'] + DL]
         + [(n, 'Persistencia') for n in ['LightGBM global'] + DL])
pruebas = pd.DataFrame([comparar(a, b) for a, b in pares])
for col in ['p Wilcoxon Brier', 'p Wilcoxon PR-AUC']:
    pruebas[col.replace('Wilcoxon', 'Holm')] = holm(pruebas[col])

D05 = os.path.dirname(PRED05)
res.round(4).to_csv(f'{D05}/metricas_globales_desde_predicciones.csv')
pruebas.to_csv(f'{D05}/pruebas_pareadas_desde_predicciones.csv', index=False)
cols = ['A', 'B', 'dif_F1 (A-B)', 'IC95_F1', 'dif_PR-AUC (A-B)', 'IC95_PR-AUC',
        'Brier mejor en A (% bloques)', 'p Holm Brier', 'p Holm PR-AUC']
with pd.option_context('display.width', 250, 'display.max_columns', 30):
    print('\nPruebas pareadas (p ajustados por Holm):')
    print(pruebas[cols].round(4).to_string(index=False))

SUBS = {'Sin 2020–2022': PR[~PR['anio_o'].between(2020, 2022)], 'Sin 2025': PR[PR['anio_o'] < 2025]}
for k, g in SUBS.items():
    t = pd.DataFrame({n: evaluar_modelo(g, n) for n in NOMBRES}).T
    print(f'\nSensibilidad — {k}:')
    print(t[['f1', 'pr_auc', 'recall_inicios', 'falsas_alarmas']].round(3).to_string())
for n in DL:
    gana = pd.Series({m: average_precision_score(g['y'], g[f'p|{n}']) > average_precision_score(g['y'], g['p|LightGBM global'])
                      for m, g in PR.groupby('mun') if g['y'].nunique() == 2})
    print(f'{n} tiene mejor PR-AUC que LightGBM en {gana.sum()} de {len(gana)} municipios')
print('\nGuardado en', D05)

39 municipios | 4,507 periodos de prueba | 335 inicios de brote
falsas_alarmas = proporción de meses SIN exceso en los que el modelo alerta
                            f1  pr_auc  recall  precision  recall_inicios  falsas_alarmas  brier
Persistencia             0.742   0.625   0.745      0.739           0.000           0.108  0.151
Binomial negativa local  0.739   0.844   0.820      0.673           0.522           0.164  0.100
LightGBM global          0.756   0.837   0.838      0.689           0.466           0.155  0.104
GRU global (1 capa)      0.762   0.855   0.800      0.727           0.367           0.124  0.102
LSTM global (2 capas)    0.748   0.824   0.845      0.671           0.475           0.171  0.112
TCN global               0.757   0.842   0.839      0.690           0.400           0.155  0.105

Pruebas pareadas (p ajustados por Holm):
                    A                       B  dif_F1 (A-B)         IC95_F1  dif_PR-AUC (A-B)      IC95_PR-AUC  Brier mejor en A (% bloques